# Construct Ontology DB from csv files
This is a starter kit to construct Ontology DB from scratch. `onto_starter_kit` folder contains the csv files to start with and this notebook to extract, transform and load.

In [1]:
import os
from dotenv import load_dotenv
from io import BytesIO
from neo4j import GraphDatabase, RoutingControl, Result
import pandas as pd
import requests
from datetime import datetime, timezone
import uuid
from to_onto import onto_nodes, onto_relations, is_source_of, instance_of, in_area, assign_attribute, assign_area, schema_nodes

In [ ]:
# load secrets from .env file
load_dotenv()
GITHUB_TOKEN = os.getenv("DOWNLOAD_ONTO_TOKEN")
GITHUB_USER = os.getenv("GITHUB_USER")
REPO_NAME = os.getenv("ONTO_REPO")
BRANCH = os.getenv("BRANCH")

## Extract csv files from github 

In [ ]:
# Raw file URL from GitHub
def download_from_github(file_name):
    url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/onto_starter_kit/{file_name}"

    # Set up authentication headers
    headers = {"Authorization": f"token {GITHUB_TOKEN}"}

    # Download and load the csv file
    response = requests.get(url, headers=headers)
    return pd.read_csv(BytesIO(response.content))

In [ ]:
asset_schema = download_from_github('asset_schema.csv')
attribute_schema = download_from_github('attribute_schema.csv')
relation_schema = download_from_github('relation_schema.csv')
asset_area_schema = download_from_github('asset_area_schema.csv')
metadata_source_schema = download_from_github('metadata_source_schema.csv')

asset_types = download_from_github('asset_types.csv')
asset_area_types = download_from_github('asset_area_types.csv')
attribute_types = download_from_github('attribute_types.csv')
relation_types = download_from_github('relation_types.csv') # with source and target asset type names
att_at = download_from_github('att_at.csv')
at_aat = download_from_github('at_aat.csv')
# concat strings for relation_types
relation_types['concat_string'] = relation_types['source_asset_type_name'] + '-' + relation_types['name'] + '-' + relation_types['target_asset_type_name']

## Add ontology data

In [4]:
now_utc = datetime.now(timezone.utc).replace(microsecond=0)
creator = 'dkg_ontology'
status = 'candidate'

In [5]:
# create namespace for ['asset_type', 'asset_area_type', 'attribute_type', 'relation_type', 'metadata_source', 'asset_area', 'status']
# For 'relation_type' 'name_space', concat a string ('concat_string') following pattern 
# 'source_type_name-relation_type_name-target_type_name', use it as 'name'
def generate_uuid(name_space, name):
    TYPE_NAMESPACE_UUID = uuid.uuid5(uuid.NAMESPACE_DNS, 'enterprise_data_knowledge_graph.com/'+ name_space)
    return str(uuid.uuid5(TYPE_NAMESPACE_UUID, name))

### Asset type

In [6]:
asset_types['id'] = asset_types['name'].apply(lambda x: generate_uuid('asset_type', x))
asset_types['created_on'] = now_utc
asset_types['last_modified_on'] = now_utc
asset_types['created_by'] = creator
asset_types['last_modified_by'] = creator
asset_types['status'] = status

### Attribute type

In [7]:
# 'data_type' should exist in df
attribute_types['id'] = attribute_types['name'].apply(lambda x: generate_uuid('attribute_type', x))
attribute_types['created_on'] = now_utc
attribute_types['last_modified_on'] = now_utc
attribute_types['created_by'] = creator
attribute_types['last_modified_by'] = creator
attribute_types['status'] = status

### Relation type

In [8]:
# 'inverse_name' should exist in df
relation_types['concat_string'] = relation_types['concat_string'].astype(str)
relation_types['id'] = relation_types['concat_string'].apply(lambda x: generate_uuid('relation_type', x))
relation_types['created_on'] = now_utc
relation_types['last_modified_on'] = now_utc
relation_types['created_by'] = creator
relation_types['last_modified_by'] = creator
relation_types['status'] = status

### Asset area type

In [ ]:
asset_area_types['id'] = asset_area_types['name'].apply(lambda x: generate_uuid('asset_area_type', x))
asset_area_types['created_on'] = now_utc
asset_area_types['last_modified_on'] = now_utc
asset_area_types['created_by'] = creator
asset_area_types['last_modified_by'] = creator
asset_area_types['status'] = status

## Load using defined function

### Nodes

In [ ]:
onto_nodes(asset_types, 'AssetType')
onto_nodes(attribute_types, 'AttributeType')
onto_nodes(asset_area_types, 'AssetAreaType')

In [ ]:
schema_nodes(asset_area_schema, 'AssetAreaSchema')
schema_nodes(asset_schema, 'AssetSchema')
schema_nodes(attribute_schema, 'AttributeSchema')
schema_nodes(relation_schema, 'RelationSchema')
schema_nodes(metadata_source_schema, 'MetadataSourceSchema')

### Relationships

In [ ]:
onto_relations(relation_types) # source asset type name and target asset type name will be removed
assign_attribute(att_at)
assign_area(at_aat)

## Next steps:
### Create csv files according to your reality
- metadata_sources.csv (instance)
- asset_areas.csv (instance)
- ms_aa.csv (-[is_source_of]-)
- aa_aat.csv (-[is_instance_of]-)
- at_aa.csv (-[in_area]-) 

Logic: `AssetType` is assigned to `AssetAreaType`. `AssetAreaType` has `AssetArea` instances. Therefore, `AssetType` can be in specific `AssetArea` instances.

### Transform

#### Metadata source

In [ ]:
# 'name', 'display_name', 'base_url' should exist in df
metadata_sources['id'] = metadata_sources['name'].apply(lambda x: generate_uuid('metadata_source', x))
metadata_sources['created_on'] = now_utc
metadata_sources['last_modified_on'] = now_utc
metadata_sources['created_by'] = creator
metadata_sources['last_modified_by'] = creator
metadata_sources['status'] = status

#### Asset area

In [ ]:
# 'type_name' should exist in df
asset_areas['id'] = asset_areas['name'].apply(lambda x: generate_uuid('asset_area', x))
asset_areas['created_on'] = now_utc
asset_areas['last_modified_on'] = now_utc
asset_areas['created_by'] = creator
asset_areas['last_modified_by'] = creator
asset_areas['status'] = status
asset_areas['type_id'] = asset_areas['type_name'].map(asset_area_types.set_index('name')['id'])

### Load

In [ ]:
onto_nodes(metadata_sources, 'MetadataSource')
onto_nodes(asset_areas, 'AssetArea')
is_source_of(ms_aa)
instance_of(aa_aat)
in_area(at_aa)